# Design an S3 lifecycle policy that transitions data through three storage classes and prove the bill projection

A media company's content library is 50 TB and growing 5 TB per month. The CTO wants to keep Standard latency for the first 30 days (when re-watch rates are high) and step the rest of the dataset down to Glacier tiers automatically without an engineer hand-moving objects each quarter. You have one session to author the bucket lifecycle, prove it is encoded correctly, and produce a bill projection finance can use to justify the migration.

You will build one S3 bucket with versioning enabled and a three-rule lifecycle policy:

- Standard to Standard-IA at day 30.
- Standard-IA to Glacier Flexible Retrieval at day 90.
- Glacier Flexible Retrieval to Glacier Deep Archive at day 180.

Then you upload a seed object and run a math-based 12-month projection that compares the lifecycle bill to a Standard-only baseline. The projection asserts the lifecycle plan is at least 50% cheaper.

**Time.** Plan for about 45 minutes. The S3 calls are seconds each; the cognitive work is choosing the right day thresholds and storage class transitions in the lifecycle form, then reading the projection math.

**Cost.** This lab costs effectively zero dollars. S3 lifecycle policies cost nothing to attach, the seed object is a few hundred bytes, and no transition actually fires during your session because the youngest age that triggers a transition is 30 days. The point of this lab is the bill projection math, not the spend.

**Interaction mode.** This lab runs in `config` mode (see `content/labs/INTERACTION-MODES.md`). Each task cell renders an ipywidgets form: dropdowns, number inputs, and an Apply button. You fill the form, click Apply, the widget calls the real S3 SDK with your selections. Validators always query S3, never the widget state.

This lab maps to AWS SAA-C03 Domain 4 (Design Cost-Optimized Architectures, 20% of exam weight).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks and ipywidgets. Pinned to a specific version per
# LAB-CREATION-BLUEPRINT.md build rules.

!pip install --quiet labrun-checks==0.6.0 ipywidgets

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Naming convention from blueprint section 12.

import atexit
import getpass
import json
import time

import boto3
from botocore.exceptions import ClientError
import ipywidgets as widgets
from IPython.display import display, clear_output

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "s3-storage-classes-and-lifecycle"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

BUCKET_NAME = None  # resolved after STS get_caller_identity
SEED_KEY = "library/seed.txt"
LIFECYCLE_RULE_ID = "labrun-three-tier-archive"
PREFIX = "library/"

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials, resolve bucket name.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"SAA labs run in {REGION} (N. Virginia).")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Region: {REGION}")

BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Bucket name resolved: {BUCKET_NAME}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, orphan scan. Versioning is enabled
# on the bucket so the cleanup helper enumerates list_object_versions and
# deletes both Versions and DeleteMarkers before delete_bucket.

CLEANUP_MANIFEST: list[CleanupResource] = [
    CleanupResource(
        type="s3_object",
        id=SEED_KEY,
        parent=BUCKET_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws s3api delete-object --bucket {BUCKET_NAME} --key {SEED_KEY}"
        ),
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _empty_versioned_bucket():
    """Delete every Version and DeleteMarker so delete_bucket can succeed."""
    s3c = boto3.client(
        "s3",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    try:
        paginator = s3c.get_paginator("list_object_versions")
        to_delete = []
        for page in paginator.paginate(Bucket=BUCKET_NAME):
            for v in page.get("Versions", []) or []:
                to_delete.append({"Key": v["Key"], "VersionId": v["VersionId"]})
            for d in page.get("DeleteMarkers", []) or []:
                to_delete.append({"Key": d["Key"], "VersionId": d["VersionId"]})
            if len(to_delete) >= 900:
                s3c.delete_objects(
                    Bucket=BUCKET_NAME, Delete={"Objects": to_delete, "Quiet": True}
                )
                to_delete = []
        if to_delete:
            s3c.delete_objects(
                Bucket=BUCKET_NAME, Delete={"Objects": to_delete, "Quiet": True}
            )
    except ClientError as e:
        if e.response.get("Error", {}).get("Code") not in ("NoSuchBucket",):
            pass


def _atexit_cleanup():
    try:
        _empty_versioned_bucket()
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom of this notebook first, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create the bucket.")

## Task 1: Create the bucket with the lab tag and enable versioning

S3 lifecycle policies can act on current versions and non-current versions separately. This lab targets current versions only, but the bucket carries versioning enabled so the policy shape stays realistic for a production library where every object has a version history.

The form below shows the bucket name (read-only; derived from the account ID so it is globally unique). Click Apply to create the bucket, tag it with `labrun:lab-slug`, and put a versioning configuration with `Status=Enabled`.

In [ ]:
# NBVAL_SKIP
# Task 1 widget: confirm bucket name and click Apply to create + tag +
# enable versioning. The validator queries S3 directly, never widget state.

bucket_label = widgets.Text(
    value=BUCKET_NAME, description="Bucket:", disabled=True,
    layout=widgets.Layout(width="500px"),
)
apply_task1 = widgets.Button(description="Apply", button_style="success")
out_task1 = widgets.Output()

def _on_apply_task1(_):
    with out_task1:
        clear_output()
        s3 = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            if REGION == "us-east-1":
                s3.create_bucket(Bucket=BUCKET_NAME)
            else:
                s3.create_bucket(
                    Bucket=BUCKET_NAME,
                    CreateBucketConfiguration={"LocationConstraint": REGION},
                )
            print(f"Created bucket {BUCKET_NAME}.")
        except ClientError as e:
            code_str = e.response.get("Error", {}).get("Code", "")
            if code_str in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
                print(f"Bucket {BUCKET_NAME} already exists; reusing it.")
            else:
                print(f"create_bucket failed: {e}")
                return

        try:
            s3.put_bucket_tagging(
                Bucket=BUCKET_NAME,
                Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
            )
            print(f"Tagged bucket with {LAB_TAG_KEY}={LAB_TAG_VALUE}.")
        except ClientError as e:
            print(f"put_bucket_tagging failed: {e}")
            return

        try:
            s3.put_bucket_versioning(
                Bucket=BUCKET_NAME,
                VersioningConfiguration={"Status": "Enabled"},
            )
            print("Versioning enabled on the bucket.")
        except ClientError as e:
            print(f"put_bucket_versioning failed: {e}")

apply_task1.on_click(_on_apply_task1)

display(widgets.VBox([
    widgets.HTML("<b>Task 1: bucket + tag + versioning</b>"),
    bucket_label,
    apply_task1,
    out_task1,
]))

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: bucket exists with the lab tag and versioning Enabled.

def checkpoint_1(session):
    try:
        s3c = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            tagging = s3c.get_bucket_tagging(Bucket=BUCKET_NAME)
        except ClientError as e:
            code_str = e.response.get("Error", {}).get("Code", "")
            if code_str in ("NoSuchBucket",):
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Bucket {BUCKET_NAME} does not exist. Click Apply on Task 1.",
                )
            if code_str in ("NoSuchTagSet",):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Bucket {BUCKET_NAME} exists but has no tags. Call "
                        f"put_bucket_tagging with {LAB_TAG_KEY}={LAB_TAG_VALUE}."
                    ),
                )
            raise
        tagset = {t["Key"]: t["Value"] for t in tagging.get("TagSet", [])}
        if tagset.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Bucket tagging is missing {LAB_TAG_KEY}={LAB_TAG_VALUE}. "
                    f"Found: {tagset!r}."
                ),
            )

        versioning = s3c.get_bucket_versioning(Bucket=BUCKET_NAME)
        if versioning.get("Status") != "Enabled":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Versioning Status is {versioning.get('Status')!r}, expected 'Enabled'. "
                    f"Run put_bucket_versioning with VersioningConfiguration.Status='Enabled'."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The Apply button already wires through three S3 calls: `create_bucket`, `put_bucket_tagging`, and `put_bucket_versioning`. If the checkpoint fails, scroll up to the widget output and read the printed messages, then click Apply again to retry the missing call.

</details>

<details><summary>Hint 2 (direction)</summary>

The tag must be set as `TagSet=[{Key: 'labrun:lab-slug', Value: 's3-storage-classes-and-lifecycle'}]`. The versioning call must use `VersioningConfiguration={'Status': 'Enabled'}` exactly (case matters). If your AWS account already has a bucket at this name, the create call returns `BucketAlreadyOwnedByYou` and the rest of the cell continues.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
s3.create_bucket(Bucket=BUCKET_NAME)
s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
s3.put_bucket_versioning(
    Bucket=BUCKET_NAME,
    VersioningConfiguration={"Status": "Enabled"},
)
```

</details>

**Wallet check.** Effectively zero. S3 bucket creation, tagging, and versioning configuration are free. You are still well inside the always-free tier.

## Task 2: Author the three-rule lifecycle policy

This is the cognitive work in this lab. The form below has three rows: each row sets a day threshold and a storage class for one transition. The defaults reflect the brief's intent (Standard-IA at 30 days, Glacier Flexible Retrieval at 90, Glacier Deep Archive at 180), but you can adjust them and see how the projection math reacts later.

The rule applies to objects under the `library/` prefix. The checkpoint validates the rule shape AWS persisted, not the form state.

Two things to know about S3 minimum-storage-duration:

- Standard-IA bills a minimum of 30 days per object. Transition any sooner and you still pay 30 days.
- Glacier Flexible Retrieval has a 90-day minimum.
- Glacier Deep Archive has a 180-day minimum.

These minimums are why a thoughtful lifecycle ladder transitions to IA at 30, Flexible at 90, and Deep Archive at 180. Anything sooner pays storage you would have paid anyway.

In [ ]:
# NBVAL_SKIP
# Task 2 widget: three transition rows + Apply button. Each row is a
# (Days, StorageClass) pair. Apply writes put_bucket_lifecycle_configuration.

STORAGE_CLASSES = [
    "STANDARD_IA",
    "ONEZONE_IA",
    "INTELLIGENT_TIERING",
    "GLACIER",
    "DEEP_ARCHIVE",
    "GLACIER_IR",
]

row1_days = widgets.IntText(value=30, description="Rule 1 day:", layout=widgets.Layout(width="250px"))
row1_class = widgets.Dropdown(options=STORAGE_CLASSES, value="STANDARD_IA",
                              description="Class:", layout=widgets.Layout(width="300px"))
row2_days = widgets.IntText(value=90, description="Rule 2 day:", layout=widgets.Layout(width="250px"))
row2_class = widgets.Dropdown(options=STORAGE_CLASSES, value="GLACIER",
                              description="Class:", layout=widgets.Layout(width="300px"))
row3_days = widgets.IntText(value=180, description="Rule 3 day:", layout=widgets.Layout(width="250px"))
row3_class = widgets.Dropdown(options=STORAGE_CLASSES, value="DEEP_ARCHIVE",
                              description="Class:", layout=widgets.Layout(width="300px"))

apply_task2 = widgets.Button(description="Apply", button_style="success")
out_task2 = widgets.Output()

def _on_apply_task2(_):
    with out_task2:
        clear_output()
        transitions = [
            {"Days": int(row1_days.value), "StorageClass": row1_class.value},
            {"Days": int(row2_days.value), "StorageClass": row2_class.value},
            {"Days": int(row3_days.value), "StorageClass": row3_class.value},
        ]
        config = {
            "Rules": [
                {
                    "ID": LIFECYCLE_RULE_ID,
                    "Status": "Enabled",
                    "Filter": {"Prefix": PREFIX},
                    "Transitions": transitions,
                }
            ]
        }
        s3 = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            s3.put_bucket_lifecycle_configuration(
                Bucket=BUCKET_NAME,
                LifecycleConfiguration=config,
            )
            print("Lifecycle configuration applied to the bucket.")
            print("Transitions written:")
            for t in transitions:
                print(f"  Day {t['Days']:>3} -> {t['StorageClass']}")
        except ClientError as e:
            print(f"put_bucket_lifecycle_configuration failed: {e}")

apply_task2.on_click(_on_apply_task2)

display(widgets.VBox([
    widgets.HTML("<b>Task 2: three-rule lifecycle policy on the library/ prefix</b>"),
    widgets.HBox([row1_days, row1_class]),
    widgets.HBox([row2_days, row2_class]),
    widgets.HBox([row3_days, row3_class]),
    apply_task2,
    out_task2,
]))

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: lifecycle policy has three transitions at 30/90/180 days
# targeting STANDARD_IA / GLACIER / DEEP_ARCHIVE.

EXPECTED_TRANSITIONS = [
    (30, "STANDARD_IA"),
    (90, "GLACIER"),
    (180, "DEEP_ARCHIVE"),
]

def checkpoint_2(session):
    try:
        s3c = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            cfg = s3c.get_bucket_lifecycle_configuration(Bucket=BUCKET_NAME)
        except ClientError as e:
            code_str = e.response.get("Error", {}).get("Code", "")
            if code_str in ("NoSuchLifecycleConfiguration",):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        "No lifecycle configuration found on the bucket. Click Apply "
                        "on Task 2 to write the three transitions."
                    ),
                )
            raise

        rules = cfg.get("Rules", [])
        enabled_rules = [r for r in rules if r.get("Status") == "Enabled"]
        if not enabled_rules:
            return CheckpointResult(
                status="fail",
                error_reason="Lifecycle configuration has no rules with Status='Enabled'.",
            )

        # Find a rule that covers library/ prefix and contains the three
        # required transitions. A rule with an empty Filter (whole bucket) also passes.
        matched = None
        for rule in enabled_rules:
            filt = rule.get("Filter", {}) or {}
            prefix = filt.get("Prefix")
            and_block = filt.get("And", {})
            and_prefix = and_block.get("Prefix") if and_block else None
            prefix_matches = (
                prefix in (None, "", PREFIX)
                or and_prefix in (None, "", PREFIX)
            )
            if not prefix_matches:
                continue
            transitions = rule.get("Transitions", [])
            pairs = {(int(t["Days"]), t["StorageClass"]) for t in transitions if "Days" in t}
            if all(exp in pairs for exp in EXPECTED_TRANSITIONS):
                matched = rule
                break

        if matched is None:
            actual_pairs = []
            for rule in enabled_rules:
                for t in rule.get("Transitions", []):
                    actual_pairs.append((t.get("Days"), t.get("StorageClass")))
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No enabled rule covers prefix {PREFIX!r} with all three "
                    f"required transitions {EXPECTED_TRANSITIONS}. Found pairs: {actual_pairs}."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

Check the three day thresholds in the form. The checkpoint wants 30, 90, and 180. It also wants the storage classes in the right order: Standard-IA, Glacier Flexible Retrieval, Glacier Deep Archive. The dropdowns name the classes using the AWS API constants (`STANDARD_IA`, `GLACIER`, `DEEP_ARCHIVE`).

</details>

<details><summary>Hint 2 (direction)</summary>

AWS distinguishes the two Glacier tiers by API constant: `GLACIER` is Glacier Flexible Retrieval and `DEEP_ARCHIVE` is Glacier Deep Archive. `GLACIER_IR` is Glacier Instant Retrieval, a different storage class entirely (millisecond retrieval, higher per-GB price). For the cost-optimized ladder this lab targets, use `GLACIER` at day 90 and `DEEP_ARCHIVE` at day 180.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Default values are already correct: Rule 1 day=30 + STANDARD_IA, Rule 2 day=90 + GLACIER, Rule 3 day=180 + DEEP_ARCHIVE. Click Apply with the defaults to write the lifecycle. If you changed any value, set them back to those three pairs before clicking Apply.

```python
s3.put_bucket_lifecycle_configuration(
    Bucket=BUCKET_NAME,
    LifecycleConfiguration={
        "Rules": [{
            "ID": LIFECYCLE_RULE_ID,
            "Status": "Enabled",
            "Filter": {"Prefix": "library/"},
            "Transitions": [
                {"Days": 30,  "StorageClass": "STANDARD_IA"},
                {"Days": 90,  "StorageClass": "GLACIER"},
                {"Days": 180, "StorageClass": "DEEP_ARCHIVE"},
            ],
        }],
    },
)
```

</details>

**Wallet check.** Still zero. Lifecycle configuration is free to attach and modify.

## Task 3: Upload a seed object to the library/ prefix

The seed exists to anchor the head_object storage-class check. S3 lifecycle is asynchronous: transitions evaluate at most once per day and the earliest day this lab targets is day 30, so the seed will stay in Standard for the duration of your session.

Click Apply to write `library/seed.txt` with a placeholder body. The validator confirms the upload landed and the object is in Standard (or has no explicit class, which S3 returns when the object is at the default Standard tier).

In [ ]:
# NBVAL_SKIP
# Task 3 widget: confirm key and Apply to upload seed.

seed_key_field = widgets.Text(
    value=SEED_KEY, description="Key:", disabled=True,
    layout=widgets.Layout(width="500px"),
)
seed_body_field = widgets.Text(
    value="placeholder seed for storage-class lab",
    description="Body:", layout=widgets.Layout(width="600px"),
)
apply_task3 = widgets.Button(description="Apply", button_style="success")
out_task3 = widgets.Output()

def _on_apply_task3(_):
    with out_task3:
        clear_output()
        s3 = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        body = seed_body_field.value.encode("utf-8")
        try:
            s3.put_object(Bucket=BUCKET_NAME, Key=SEED_KEY, Body=body)
            print(f"Uploaded s3://{BUCKET_NAME}/{SEED_KEY} ({len(body)} bytes).")
            head = s3.head_object(Bucket=BUCKET_NAME, Key=SEED_KEY)
            sc = head.get("StorageClass", "STANDARD (default)")
            print(f"head_object StorageClass: {sc}")
        except ClientError as e:
            print(f"put_object failed: {e}")

apply_task3.on_click(_on_apply_task3)

display(widgets.VBox([
    widgets.HTML("<b>Task 3: upload seed object</b>"),
    seed_key_field,
    seed_body_field,
    apply_task3,
    out_task3,
]))

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: seed object exists and reports StorageClass=STANDARD or omitted.

def checkpoint_3(session):
    try:
        s3c = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            head = s3c.head_object(Bucket=BUCKET_NAME, Key=SEED_KEY)
        except ClientError as e:
            code_str = e.response.get("Error", {}).get("Code", "")
            if code_str in ("404", "NoSuchKey", "NotFound"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"No object at s3://{BUCKET_NAME}/{SEED_KEY}. Click Apply "
                        f"on Task 3 to upload the seed."
                    ),
                )
            raise
        sc = head.get("StorageClass")
        # S3 omits StorageClass when the object is in Standard (the default).
        if sc not in (None, "STANDARD"):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Seed object's StorageClass is {sc!r}, expected STANDARD or omitted. "
                    f"Lifecycle transitions are asynchronous and do not fire within the session, "
                    f"so the upload should still be at Standard."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Click Apply on the Task 3 form. The widget calls `put_object` with the prefilled key (`library/seed.txt`) and a small placeholder body, then runs `head_object` and prints the StorageClass.

</details>

<details><summary>Hint 2 (direction)</summary>

If `head_object` returns 404, the upload did not land. Re-check the bucket exists (Task 1 Apply ran successfully). If StorageClass is anything other than STANDARD or absent, the put_object call set a non-default class explicitly via `StorageClass=...`, which this lab does not want. Don't pass that argument.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
s3.put_object(
    Bucket=BUCKET_NAME,
    Key="library/seed.txt",
    Body=b"placeholder seed for storage-class lab",
)
head = s3.head_object(Bucket=BUCKET_NAME, Key="library/seed.txt")
```

</details>

**Wallet check.** One S3 PUT plus one S3 HEAD, both metered at fractions of a penny per 1000 calls. Still about zero cents.

## Task 4: Run the 12-month bill projection and prove the lifecycle is 50%+ cheaper

The projection holds the dataset size constant at 50 TB across 12 months and integrates monthly spend two ways:

- **Standard-only baseline.** 50 TB at $0.023/GB-month for 12 months.
- **Lifecycle plan.** The first 30 days are at Standard at $0.023/GB-month. Days 30-89 sit in Standard-IA at $0.0125/GB-month. Days 90-179 sit in Glacier Flexible Retrieval at $0.0036/GB-month. Days 180+ sit in Glacier Deep Archive at $0.00099/GB-month.

Prices are us-east-1 published rates at the time this brief was written. Update the constants in the code if AWS changes them.

Click Apply to run the projection. The widget prints the monthly breakdown for the lifecycle plan, the totals for both plans, and the savings ratio.

In [ ]:
# NBVAL_SKIP
# Task 4 widget: dataset size + Apply -> 12-month projection.

# us-east-1 published prices at brief authoring time. Update if AWS changes.
PRICE_STANDARD_PER_GB_MONTH = 0.023
PRICE_STANDARD_IA_PER_GB_MONTH = 0.0125
PRICE_GLACIER_FLEX_PER_GB_MONTH = 0.0036
PRICE_DEEP_ARCHIVE_PER_GB_MONTH = 0.00099

dataset_tb_field = widgets.IntText(
    value=50, description="Dataset (TB):", layout=widgets.Layout(width="300px"),
)
apply_task4 = widgets.Button(description="Apply", button_style="success")
out_task4 = widgets.Output()
PROJECTION_STATE = {"standard_only_total": None, "lifecycle_total": None}


def _project_lifecycle(dataset_gb):
    # Average price per GB-month for each calendar month, then sum 12 months.
    # Month 1: 30 days at Standard.
    # Months 2-3 (days 30-89): Standard-IA.
    # Months 4-6 (days 90-179): Glacier Flexible Retrieval.
    # Months 7-12 (days 180+): Glacier Deep Archive.
    months = []
    months.append(("Month  1 (Standard)        ", PRICE_STANDARD_PER_GB_MONTH))
    months.append(("Month  2 (Standard-IA)     ", PRICE_STANDARD_IA_PER_GB_MONTH))
    months.append(("Month  3 (Standard-IA)     ", PRICE_STANDARD_IA_PER_GB_MONTH))
    months.append(("Month  4 (Glacier Flex)    ", PRICE_GLACIER_FLEX_PER_GB_MONTH))
    months.append(("Month  5 (Glacier Flex)    ", PRICE_GLACIER_FLEX_PER_GB_MONTH))
    months.append(("Month  6 (Glacier Flex)    ", PRICE_GLACIER_FLEX_PER_GB_MONTH))
    for i in range(7, 13):
        months.append((f"Month {i:>2} (Deep Archive)    ", PRICE_DEEP_ARCHIVE_PER_GB_MONTH))
    total = 0.0
    rows = []
    for label, ppgbm in months:
        spend = dataset_gb * ppgbm
        rows.append((label, ppgbm, spend))
        total += spend
    return rows, total


def _on_apply_task4(_):
    with out_task4:
        clear_output()
        dataset_gb = int(dataset_tb_field.value) * 1024
        standard_only = dataset_gb * PRICE_STANDARD_PER_GB_MONTH * 12
        rows, lifecycle_total = _project_lifecycle(dataset_gb)

        print(f"Dataset: {dataset_tb_field.value} TB ({dataset_gb} GB)")
        print(f"Standard-only baseline (12 months): ${standard_only:,.2f}")
        print()
        print("Lifecycle plan monthly breakdown:")
        for label, ppgbm, spend in rows:
            print(f"  {label} @ ${ppgbm:.5f}/GB-month -> ${spend:,.2f}")
        print()
        print(f"Lifecycle 12-month total: ${lifecycle_total:,.2f}")
        if standard_only > 0:
            savings = (standard_only - lifecycle_total) / standard_only
            print(f"Savings vs Standard-only: {savings*100:.1f}%")
        PROJECTION_STATE["standard_only_total"] = standard_only
        PROJECTION_STATE["lifecycle_total"] = lifecycle_total

apply_task4.on_click(_on_apply_task4)

display(widgets.VBox([
    widgets.HTML("<b>Task 4: 12-month bill projection</b>"),
    dataset_tb_field,
    apply_task4,
    out_task4,
]))

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: lifecycle total is at least 50% cheaper than Standard-only.

def checkpoint_4(session):
    try:
        standard_only = PROJECTION_STATE.get("standard_only_total")
        lifecycle = PROJECTION_STATE.get("lifecycle_total")
        if standard_only is None or lifecycle is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Projection not run yet. Click Apply on Task 4 with the default "
                    "50 TB dataset and inspect the output before this checkpoint."
                ),
            )
        if standard_only <= 0:
            return CheckpointResult(
                status="fail",
                error_reason="Standard-only baseline is zero. Re-run Task 4 with a non-zero dataset size.",
            )
        ratio = lifecycle / standard_only
        if ratio >= 0.5:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Lifecycle plan came in at {ratio*100:.1f}% of the Standard-only baseline; "
                    f"target is below 50%. If the projection prices were changed, restore them "
                    f"to the published us-east-1 rates documented above the projection function."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

Click Apply on the Task 4 form. The default dataset size (50 TB) and the default ladder defaults are wired to produce a savings ratio well above 50%. The numbers print in the widget output.

</details>

<details><summary>Hint 2 (direction)</summary>

The Standard-only baseline is `dataset_gb * $0.023/GB-month * 12 months`. The lifecycle plan integrates four price tiers across the year: 1 month at Standard ($0.023), 2 months at Standard-IA ($0.0125), 3 months at Glacier Flexible ($0.0036), 6 months at Glacier Deep Archive ($0.00099). The savings come from the bottom of the ladder: Deep Archive is roughly 23x cheaper than Standard per GB-month.

</details>

<details><summary>Hint 3 (spoiler)</summary>

At 50 TB:

- Standard-only 12-month: 51,200 GB x $0.023 x 12 = $14,131.20.
- Lifecycle: ~ $14,131 / 12 (one Standard month) + $640 x 2 (Standard-IA) + $184 x 3 (Glacier Flex) + $50.7 x 6 (Deep Archive) = ~$1,177 + $1,280 + $552 + $304 = ~$3,313.
- Ratio: ~23%, well under 50%.

The default form values produce these numbers. Just click Apply.

</details>

**Wallet check.** Math only, no API calls beyond what you already paid for. Still about zero cents.

## Cleanup

The bucket has versioning enabled so the cleanup helper enumerates `list_object_versions` and deletes every Version and DeleteMarker before `delete_bucket` runs. Re-running this cell is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Empty the versioned bucket first, then run_cleanup deletes the
# bucket itself (s3_bucket adapter calls delete_bucket).

import sys

_empty_versioned_bucket()
print("Emptied bucket of all object versions and delete markers.")

result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
critical_destroyed = 0
standard_total = len(CLEANUP_MANIFEST)
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.00.** A single seed object at a few bytes plus a handful of S3 PUT and HEAD calls fits well under the always-free tier. The lifecycle policy and the bill projection cost nothing to author.

## Reflection

These are not graded. They are for you.

1. Walk through what happens to a single S3 object's monthly bill across its first 12 months under a Standard to Standard-IA at 30 days to Glacier Flexible Retrieval at 90 days to Glacier Deep Archive at 180 days lifecycle. Where do the storage-class minimum-duration rules show up in the math, and what would happen to the bill if you tried to transition objects after only 15 days at Standard?

2. Your team is debating whether to use `Standard-IA` or `Intelligent-Tiering` for a dataset with unpredictable access patterns. Walk through how each storage class bills (storage cost, retrieval cost, monitoring fee, minimum-duration penalty), and explain when each is the right call.

## Exam-Style Practice

**Question 1 (Domain 4, storage-class minimum-duration rules):**

A media team uploads 1 TB of new video assets per week to S3 Standard. They configure a lifecycle rule that transitions any object older than 7 days to S3 Standard-IA. After running for one quarter, the finance team notices the Standard-IA bill is higher than projected for objects that were deleted before day 30. What is the most likely cause?

A. Standard-IA bills a minimum of 30 days per object, so objects deleted before day 30 still incur 30 days of Standard-IA storage charges.

B. Standard-IA bills a retrieval fee on every transition from Standard, which compounds as the lifecycle rule fires daily.

C. Standard-IA replicates objects to a second Region by default, doubling the per-GB price.

D. Standard-IA cannot be used with versioning, so the bucket silently fell back to Standard.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: S3 Standard-IA, One Zone-IA, Glacier Flexible Retrieval, and Glacier Deep Archive all have minimum-storage-duration rules. Standard-IA's minimum is 30 days. Deleting or transitioning an object earlier than its minimum-duration window still pays the storage fee through the minimum. This is the most-tested storage-class trap on the SAA-C03 exam.
- B is wrong: lifecycle transitions from Standard to Standard-IA cost a small per-1000-objects fee, not a retrieval fee. Retrieval fees apply to GETs against IA and Glacier classes.
- C is wrong: Standard-IA stores data in a single Region across multiple AZs, the same regional model as Standard. Cross-Region replication is opt-in via separate configuration.
- D is wrong: versioning and Standard-IA work together; lifecycle rules can target current versions, non-current versions, or both.

</details>

**Question 2 (Domain 4, storage-class selection for unpredictable access):**

You are storing log files that are accessed frequently for the first 30 days, then rarely (but with unpredictable spikes for compliance audits) for the next 5 years. Audits happen 0-3 times per year and pull random objects from any age. Which S3 storage class is the best fit for the 30-day-plus window?

A. Glacier Deep Archive, because the price is lowest and audits are rare.

B. Standard-IA, because IA tolerates the unpredictable retrieval pattern without the multi-hour Glacier retrieval delay.

C. Intelligent-Tiering, because the workload's access pattern is unpredictable in both timing and which objects get accessed.

D. Standard, because audit retrievals must complete in milliseconds.

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong: Glacier Deep Archive has a 12-hour standard retrieval delay (Bulk tier) or a $0.03/GB Expedited tier that still takes 1-5 minutes. An auditor pulling a random object cannot wait 12 hours; choosing Deep Archive trades the wrong constraint.
- B is partially correct: Standard-IA is fine for retrieval latency, but it bills a per-GB retrieval fee on every audit GET. Across 5 years of unpredictable access patterns the retrieval fees can dwarf the storage savings.
- C is correct: Intelligent-Tiering monitors access patterns per object and moves objects between Frequent, Infrequent, and Archive tiers automatically. It charges a small per-object monitoring fee but eliminates the retrieval-fee gamble. AWS published guidance recommends Intelligent-Tiering as the default choice for any workload with unpredictable access.
- D is wrong: Standard is the right pick for the first 30 days, but it is the most expensive option for storage at rest. The question asks about the 30-day-plus window, where Standard is overkill.

</details>